In [1]:
!pip install -q transformers peft trl bitsandbytes accelerate datasets wandb

In [2]:
from huggingface_hub import login as hf_login
import wandb

print("🔑 Logging into Hugging Face Hub...")
hf_login()

print("📊 Logging into Weights & Biases...")
wandb.login()

🔑 Logging into Hugging Face Hub...
📊 Logging into Weights & Biases...


wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /root/.netrc.
wandb: Currently logged in as: lithmanlisara04 (lithmanlisara04-sliit) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [3]:
import os
import torch
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, prepare_model_for_kbit_training
from trl import SFTTrainer, SFTConfig

# Base Model and Hugging Face Destinations
BASE_MODEL_ID = "meta-llama/Meta-Llama-3.1-8B-Instruct"
DATASET_REPO = "Lisara/desktriage-dataset"
ADAPTER_REPO_ID = "Lisara/desktriage-adapters"
LOCAL_OUTPUT_DIR = "./hf_adapters_8b"

In [4]:
print(f"📡 Loading private dataset from HF Hub: {DATASET_REPO}...")
dataset = load_dataset(DATASET_REPO, token=True)

# Extract train split
if "train" in dataset:
    dataset_split = dataset["train"]
else:
    dataset_split = dataset

# Split into train/validation (90/10) to monitor loss during training
dataset_split = dataset_split.train_test_split(test_size=0.1, seed=42)
print(f"✅ Dataset loaded. Train size: {len(dataset_split['train'])}, Val size: {len(dataset_split['test'])}")

📡 Loading private dataset from HF Hub: Lisara/desktriage-dataset...
✅ Dataset loaded. Train size: 847, Val size: 95


In [5]:
# Configure 4-bit Quantization (bitsandbytes QLoRA)
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True
)

# Load Tokenizer
print(f"🔄 Loading Tokenizer: {BASE_MODEL_ID}...")
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_ID, token=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

# Load Base Model
print(f"🔄 Loading Base Model: {BASE_MODEL_ID}...")
model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.bfloat16,
    token=True
)

# Prepare model for k-bit training (activates gradient checkpointing)
model = prepare_model_for_kbit_training(model)

🔄 Loading Tokenizer: meta-llama/Meta-Llama-3.1-8B-Instruct...


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


🔄 Loading Base Model: meta-llama/Meta-Llama-3.1-8B-Instruct...


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


In [6]:
# Define LoRA parameters (SFTTrainer will attach this internally in Cell 7)
peft_config = LoraConfig(
    r=8,
    lora_alpha=20,
    target_modules=["q_proj", "v_proj", "k_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)
print("✅ LoRA configuration defined.")

✅ LoRA configuration defined.


In [7]:
# Set up SFTConfig (replaces TrainingArguments in the latest TRL library)
sft_config = SFTConfig(
    output_dir=LOCAL_OUTPUT_DIR,
    dataset_text_field="text",     # Handled inside SFTConfig
    max_length=1024,               # Handled inside SFTConfig (replaces max_seq_length)
    packing=False,                 # Handled inside SFTConfig
    per_device_train_batch_size=2,
    gradient_accumulation_steps=2, # Effective batch size = 4
    learning_rate=5e-5,            # Stabilized learning rate matching MLX
    logging_steps=10,
    max_steps=400,                 # Total iterations matching our local setup
    save_strategy="steps",
    save_steps=100,
    eval_strategy="steps",         # Changed from evaluation_strategy
    eval_steps=100,
    fp16=False,
    bf16=True,                     # Use bfloat16 precision for training stability
    optim="paged_adamw_8bit",      # High-efficiency memory optimizer
    warmup_steps=12,               # Fixed from warmup_ratio deprecation
    lr_scheduler_type="constant",  # Matches MLX schedule
    report_to="wandb"              # Logs graphs directly to Wandb
)

# Initialize Supervised Fine-Tuning Trainer
trainer = SFTTrainer(
    model=model,
    train_dataset=dataset_split["train"],
    eval_dataset=dataset_split["test"],
    peft_config=peft_config,       # Passed here for internal wrapping
    processing_class=tokenizer,    # Replaces tokenizer=tokenizer in latest TRL
    args=sft_config
)

print("✅ Model wrapped with LoRA adapters successfully.")
trainer.model.print_trainable_parameters()

/tmp/ipykernel_6457/2838358925.py:2: FutureWarning: The default `loss_type` will change from `'nll'` to `'chunked_nll'` in TRL 1.7. For standard models this is transparent (same math, lower memory) and no action is needed — you'll get the new default automatically on upgrade. If you use a custom model, check ahead of time that `loss_type='chunked_nll'` runs and yields the same loss as `'nll'`; if it doesn't, pin `loss_type='nll'` to keep the current behavior and please open an issue at https://github.com/huggingface/trl/issues so we can address the edge case.
  sft_config = SFTConfig(


Adding EOS to train dataset:   0%|          | 0/847 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/847 [00:00<?, ? examples/s]

Adding EOS to eval dataset:   0%|          | 0/95 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/95 [00:00<?, ? examples/s]

✅ Model wrapped with LoRA adapters successfully.
trainable params: 20,971,520 || all params: 8,051,232,768 || trainable%: 0.2605


In [8]:
print("🔥 Starting Model Training Run...")
trainer.train()

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 128009, 'pad_token_id': 128009}.


🔥 Starting Model Training Run...


Step,Training Loss,Validation Loss,Entropy,Mean Token Accuracy,Num Tokens
100,1.001579,0.984711,0.992958,0.722317,74133.000000
200,0.923512,0.938366,0.933763,0.730341,146007.000000
300,0.851488,0.921193,0.884952,0.731459,219770.000000
400,0.797294,0.902983,0.853018,0.737291,293876.000000


TrainOutput(global_step=400, training_loss=0.9960796403884887, metrics={'train_runtime': 9551.5331, 'train_samples_per_second': 0.168, 'train_steps_per_second': 0.042, 'total_flos': 1.5433340269142016e+16, 'train_loss': 0.9960796403884887, 'epoch': 1.8867924528301887})

In [9]:
# 1. Save adapters locally in Colab
print(f"💾 Saving fine-tuned adapters locally to {LOCAL_OUTPUT_DIR}...")
trainer.model.save_pretrained(LOCAL_OUTPUT_DIR)
tokenizer.save_pretrained(LOCAL_OUTPUT_DIR)

# 2. Push directly to your Hugging Face Hub (Private Repo)
print(f"🛰️ Uploading adapters to Hugging Face Hub: {ADAPTER_REPO_ID}...")
try:
    trainer.model.push_to_hub(
        repo_id=ADAPTER_REPO_ID,
        token=True,
        private=True
    )
    tokenizer.push_to_hub(
        repo_id=ADAPTER_REPO_ID,
        token=True,
        private=True
    )
    print(f"🎉 Success! Adapters are safely stored in your Hugging Face Hub.")
    print(f"🔗 Hub Destination: https://huggingface.co/models/{ADAPTER_REPO_ID}")
except Exception as e:
    print(f"❌ Failed to upload adapters to Hugging Face Hub: {str(e)}")

💾 Saving fine-tuned adapters locally to ./hf_adapters_8b...
🛰️ Uploading adapters to Hugging Face Hub: Lisara/desktriage-adapters...


Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...adapter_model.safetensors:   0%|          | 53.2kB / 42.0MB            

README.md:   0%|          | 0.00/5.17k [00:00<?, ?B/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...mpoajwzg33/tokenizer.json: 100%|##########| 17.2MB / 17.2MB            

🎉 Success! Adapters are safely stored in your Hugging Face Hub.
🔗 Hub Destination: https://huggingface.co/models/Lisara/desktriage-adapters
